In [ ]:
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler , LabelEncoder , OneHotEncoder
import pickle 

In [ ]:
#Load the dataset 
df = pd.read_csv('Churn_Modelling.csv')
df.head()

In [ ]:
df = df.drop(columns = ['RowNumber' , 'CustomerId' , 'Surname'] , axis = 1) 
df

In [ ]:
#Encode categorical variables
label_encoder_gender = LabelEncoder()

df['Gender'] = label_encoder_gender.fit_transform(df['Gender']) 
df

In [ ]:
#OneHot encode 'Geography' column 
encoder = OneHotEncoder(sparse_output=False) 
encoded = encoder.fit_transform(df[['Geography']]) 

#Create a dataframe with encoded columns 
encoded_df = pd.DataFrame(encoded , columns = encoder.get_feature_names_out(['Geography']) , index = df.index)

#Concatenate and drop original column 
df = pd.concat([df.drop(columns = ['Geography']) , encoded_df] , axis=1)

In [ ]:
#Save the encoder and scaler
with open('label_encoder_gender.pkl' , 'wb') as file:
    pickle.dump(label_encoder_gender , file) 
    
with open('onehot_encoder_geo.pkl' , 'wb') as file:
    pickle.dump(encoder , file)     

In [ ]:
#Divide the dataframe into independent and dependent features
X = df.drop('Exited' , axis=1) 
y = df['Exited']

#Split the data in training and testing sets
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

#Scale these features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) 
X_test = scaler.transform(X_test) 

In [ ]:
with open('scaler.pkl' , 'wb') as file:
    pickle.dump(scaler , file)
    

# ANN Implementation

In [ ]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense 
from tensorflow.keras.callbacks import EarlyStopping , TensorBoard 
import datetime

In [ ]:
#Building ANN Model 
model = Sequential([
    Dense(64 , activation = 'relu' , input_shape = (X_train.shape[1],)) , #Hidden layer 1 connected with input layer
    Dense(32 , activation = 'relu') ,#Hidden layer 2  
    Dense(1 , activation='sigmoid') #output layer
]
    
)

In [ ]:
model.summary()

In [ ]:
#Compile the model 
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.01) 
loss = tensorflow.keras.losses.BinaryCrossentropy()
loss

In [ ]:
#Compile the model 
model.compile(optimizer=opt , loss = 'binary_crossentropy' , metrics = ['accuracy'])

In [ ]:
#Setup the Tensorboard 
from tensorflow.keras.callbacks import EarlyStopping , TensorBoard

log_dir = 'logs/fit' + datetime.datetime.now().strftime("%Y%M%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir = log_dir , histogram_freq = 1)

In [ ]:
#Setup up Early Stopping 
early_stopping_callback = EarlyStopping(monitor = 'val_loss' , patience = 5 , restore_best_weights = True)

In [ ]:
#Train the model 
history = model.fit(
    X_train , y_train , validation_data = (X_test , y_test) , epochs = 100 , 
    callbacks = [tensorflow_callback , early_stopping_callback]
)

In [ ]:
model.save('model.h5') 

In [ ]:
#Load Tensorboard Extension 
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit